# Business Entity Resolution: v4 (embedding search shortlist)

**Settings before running** (right-hand panel, *Session options*): **Accelerator = GPU T4 x2**, **Internet = On**
(Internet is needed once, to download the small pretrained text model). Do not pick GPU P100: recent PyTorch no longer supports it.

**How to run:** click **Save Version** (top right), choose **Save & Run All (Commit)**, then **Save**. It runs on Kaggle's servers even if you close the tab.

What is new compared with v1:
1. v1 only looked for matches **inside the same state**, and kept the top 15 by name and top 15 by address. About 2.3% of true matches never made that shortlist.
2. v4 adds two country-wide searches on top:
   - **Embeddings:** a small pretrained model (BAAI/bge-small-en-v1.5, MIT licence, 33M parameters) turns "name , address" into 384 numbers. Similar businesses get similar numbers, and the GPU finds the 20 closest records in the whole country.
   - **Spelling search:** character 3-gram TF-IDF of name + address (catches heavy misspellings like "praaivett limittedd"), 30 closest per record.
3. All candidates are merged, and LightGBM scores them with v1's features plus the new similarity numbers.

Cells: 1 GPU check, 2 install, 3 v1 pipeline code, 4 settings, 5 v4 code, 6 **train** (prints `shortlist recall` and `BEST ... val F0.5`; v1 was recall 0.9765, F0.5 0.9529), 7 **predict** test and write `output/matching_results.tsv`, 8 official format check.

No external data or APIs are used, only the competition files and the pretrained model weights.

In [ ]:
!nvidia-smi -L || echo 'NO GPU FOUND: set Accelerator to GPU T4 x2 (the notebook still runs on CPU, but far slower)'
!free -g

In [ ]:
!pip install -q rapidfuzz sparse_dot_topn unidecode polars lightgbm

In [ ]:
# =====================================================================
# Business Entity Resolution: full pipeline (clean -> shortlist -> features -> LightGBM -> decide)
# No external data or services are used anywhere. Only the provided files.
# =====================================================================
import os, re, gc, glob, time, json
import numpy as np
import pandas as pd
import polars as pl
from multiprocessing import Pool
from unidecode import unidecode
from sklearn.feature_extraction.text import TfidfVectorizer
from sparse_dot_topn import sp_matmul_topn
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler
from rapidfuzz.process import cpdist
import lightgbm as lgb

N_CPU = os.cpu_count() or 4

def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)

# ---------------------------------------------------------------------
# 1. Loading. Files are tab-separated, all columns read as text.
#    Handles files split into *_part_N.tsv (BOM / CRLF / repeated header).
# ---------------------------------------------------------------------
def find_file(root, name):
    hits = sorted(glob.glob(f"{root}/**/{name}.tsv", recursive=True))
    if hits:
        return [hits[0]]
    parts = glob.glob(f"{root}/**/{name}_part_*.tsv", recursive=True)
    parts.sort(key=lambda p: int(re.findall(r"_part_(\d+)", p)[-1]))
    if not parts:
        raise FileNotFoundError(f"{name}.tsv not found under {root}")
    return parts

def load(root, name):
    frames, header = [], None
    for p in find_file(root, name):
        txt = open(p, encoding="utf-8-sig").read().replace("\r\n", "\n").replace("\r", "")
        if txt.startswith(("entity_id\t", "source1_entity_id\t")):
            header, txt = txt.split("\n", 1)
        frames.append(txt)
    body = "".join(f if f.endswith("\n") else f + "\n" for f in frames)
    df = pl.read_csv((header + "\n" + body).encode(), separator="\t", quote_char=None, infer_schema_length=0)
    return df.fill_null("").to_pandas()

# ---------------------------------------------------------------------
# 2. Cleaning (normalisation). Hand-written rules only.
# ---------------------------------------------------------------------
ABBR = {"rd":"road","st":"street","ave":"avenue","av":"avenue","blvd":"boulevard","bd":"boulevard","hwy":"highway",
        "dr":"drive","ln":"lane","ct":"court","pl":"place","pkwy":"parkway","cir":"circle","ste":"suite","fl":"floor",
        "apt":"apartment","no":"number","nr":"near","opp":"opposite","bldg":"building","ind":"industrial",
        "dist":"district","r":"rue","ch":"chemin","n":"north","s":"south","e":"east","w":"west","mt":"mount",
        "ft":"fort","hts":"heights","sq":"square","imp":"impasse","all":"allee"}
NULLS = {"null", "na", "none", "nan"}
# legal words, written in "skeleton" form (see skel_tok): covers Limited/Ltd/limittedd, Private/Pvt/praaivett, LLC, SARL ...
LEGAL_SK = {"lmtd","ltd","prwt","pwt","llc","llp","inc","ncrprtd","crp","crprtn","c","cmpny","plc","srl","ss","sa",
            "sc","erl","snc","gmbh","t","m","s","d","dba","lp","pc","pllc","pa","ms","and"}

def base(s):
    s = unidecode(str(s)).lower()                                   # also turns Hindi/Telugu/... script into Latin letters
    s = re.sub(r"<?null>?|n/a", " ", s)
    s = re.sub(r"https?://|www\.|\.(com|net|org|in|co|io|biz|fr|us)\b", " ", s)
    s = s.replace("&", " and ").replace("+", " and ")
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def skel_tok(t):
    """Spelling-robust 'skeleton' of a word: drop vowels, merge ph/f, bh/b ..., collapse double letters.
    'limited' and the transliterated 'limittedd' both become 'lmtd'."""
    if not t or t.isdigit():
        return t
    t = t.replace("0", "o").replace("1", "l") if re.search(r"[a-z]", t) else t   # leetspeak: C1ean -> Clean
    t = re.sub(r"ph", "f", t)
    t = re.sub(r"([bkgtdcs])h", r"\1", t)
    t = t.replace("v", "w").replace("q", "k").replace("z", "s").replace("x", "ks")
    t = t[0] + re.sub(r"[aeiouy]", "", t[1:])
    return re.sub(r"(.)\1+", r"\1", t)

US = {"alabama":"al","alaska":"ak","arizona":"az","arkansas":"ar","california":"ca","colorado":"co","connecticut":"ct",
"delaware":"de","florida":"fl","georgia":"ga","hawaii":"hi","idaho":"id","illinois":"il","indiana":"in","iowa":"ia",
"kansas":"ks","kentucky":"ky","louisiana":"la","maine":"me","maryland":"md","massachusetts":"ma","michigan":"mi",
"minnesota":"mn","mississippi":"ms","missouri":"mo","montana":"mt","nebraska":"ne","nevada":"nv","new hampshire":"nh",
"new jersey":"nj","new mexico":"nm","new york":"ny","north carolina":"nc","north dakota":"nd","ohio":"oh","oklahoma":"ok",
"oregon":"or","pennsylvania":"pa","rhode island":"ri","south carolina":"sc","south dakota":"sd","tennessee":"tn",
"texas":"tx","utah":"ut","vermont":"vt","virginia":"va","washington":"wa","west virginia":"wv","wisconsin":"wi",
"wyoming":"wy","district of columbia":"dc","puerto rico":"pr"}
IN = {"andhra pradesh":"ap","arunachal pradesh":"ar","assam":"as","bihar":"br","chhattisgarh":"cg","goa":"ga",
"gujarat":"gj","haryana":"hr","himachal pradesh":"hp","jharkhand":"jh","karnataka":"ka","kerala":"kl",
"madhya pradesh":"mp","maharashtra":"mh","manipur":"mn","meghalaya":"ml","mizoram":"mz","nagaland":"nl","odisha":"od",
"orissa":"od","punjab":"pb","rajasthan":"rj","sikkim":"sk","tamil nadu":"tn","telangana":"ts","tripura":"tr",
"uttar pradesh":"up","uttarakhand":"uk","uttaranchal":"uk","west bengal":"wb","delhi":"dl","new delhi":"dl",
"jammu and kashmir":"jk","ladakh":"la","puducherry":"py","pondicherry":"py","chandigarh":"ch",
"andaman and nicobar islands":"an","lakshadweep":"ld"}
MERGE_ST = {"ts": "ap"}            # Telangana was split from Andhra Pradesh; the data mixes them
_STATE = {}
for _d in (US, IN):
    for _full, _code in _d.items():
        _STATE[tuple(skel_tok(t) for t in _full.split())] = _code
_CODES = set(US.values()) | set(IN.values())

def states_of(raw_addr):
    """All state/province codes mentioned in an address ('' when none). Country-agnostic: unknown countries get ''."""
    found = set()
    for seg in unidecode(str(raw_addr)).lower().split(","):
        toks = re.sub(r"[^a-z ]", " ", seg).split()
        if not toks:
            continue
        if len(toks) == 1 and toks[0] in _CODES:
            found.add(toks[0]); continue
        sk = [skel_tok(t) for t in toks]
        for n in (4, 3, 2, 1):
            for i in range(len(sk) - n + 1):
                key = tuple(sk[i:i + n])
                if key in _STATE and (n > 1 or len(key[0]) >= 3):
                    found.add(_STATE[key])
    return "|".join(sorted({MERGE_ST.get(x, x) for x in found}))

def _clean_rows(rows):
    out = []
    for name, addr in rows:
        nb = base(name)
        sk = [skel_tok(t) for t in nb.split()]
        core = [t for t in sk if t not in LEGAL_SK] or sk
        an = " ".join(ABBR.get(t, t) for t in base(addr).split() if t not in NULLS)
        out.append((nb, " ".join(core), "".join(core), an,
                    " ".join(skel_tok(t) for t in an.split()), states_of(addr)))
    return out

def prepare(df):
    """Adds the cleaned columns. Runs on all CPU cores."""
    rows = list(zip(df.business_name.tolist(), df.business_address.tolist()))
    step = 50_000
    with Pool(N_CPU) as pool:
        res = pool.map(_clean_rows, [rows[i:i + step] for i in range(0, len(rows), step)])
    res = [r for chunk in res for r in chunk]
    cols = ["name_b", "name_sk", "name_skc", "addr_n", "addr_sk", "states"]
    for j, c in enumerate(cols):
        df[c] = [r[j] for r in res]
    df.drop(columns=["business_address"], inplace=True)
    df["nonascii"] = df.business_name.str.contains(r"[^\x00-\x7f]", regex=True).astype(np.int8)
    df.drop(columns=["business_name"], inplace=True)
    return df.reset_index(drop=True)

# ---------------------------------------------------------------------
# 3. Shortlist ("blocking"). For each Source 1 record, look only inside the same country and
#    the same state, and keep the top-k most similar names and the top-k most similar addresses.
#    Records whose state could not be read form a separate pool per country.
# ---------------------------------------------------------------------
def _vec_name():
    return TfidfVectorizer(analyzer="char", ngram_range=(3, 3), dtype=np.float32, sublinear_tf=True)
def _vec_addr():
    return TfidfVectorizer(analyzer="word", token_pattern=r"\S+", ngram_range=(1, 2), dtype=np.float32, sublinear_tf=True)

class Index:
    """TF-IDF indexes over one pool of candidate records (one country of Source 2 or Source 3)."""
    def __init__(self, B, k_state=15, k_unk=10):
        self.B, self.k_state, self.k_unk = B, k_state, k_unk
        self.cache = {}
        st = B.states.str.split("|")
        self.members = {}
        for pos, lst in zip(range(len(B)), st):
            for s in lst:
                if s:
                    self.members.setdefault(s, []).append(pos)
        self.unknown = np.flatnonzero(B.states.to_numpy() == "")

    def _build(self, key, pos):
        if key in self.cache:
            return self.cache[key]
        pos = np.asarray(pos, dtype=np.int64)
        entry = {"pos": pos}
        for col, mk in (("name_skc", _vec_name), ("addr_sk", _vec_addr)):
            try:
                v = mk(); M = v.fit_transform(self.B[col].to_numpy()[pos])
                entry[col] = (v, M.T.tocsr())
            except ValueError:            # empty vocabulary, e.g. every address blank
                entry[col] = None
        self.cache[key] = entry
        return entry

    def query(self, A, thr=0.1, chunk=40_000):
        """Returns DataFrame(a, b, sim_name, sim_addr); a = row in A, b = row in self.B."""
        out = []
        ast = A.states.str.split("|").tolist()
        groups = {}
        for i, lst in enumerate(ast):
            for s in lst:
                groups.setdefault(s, []).append(i)
        jobs = []
        for s, rows in groups.items():
            if s == "":
                jobs.append(("__all__", np.arange(len(self.B)), rows, self.k_state))
            elif s in self.members:
                jobs.append((s, self.members[s], rows, self.k_state))
        known = [i for i, lst in enumerate(ast) if lst != [""]]
        if len(self.unknown) and known:
            jobs.append(("__unknown__", self.unknown, known, self.k_unk))
        for key, pos, rows, k in jobs:
            e = self._build(key, pos)
            rows = np.asarray(rows)
            for col, tag in (("name_skc", 0), ("addr_sk", 1)):
                if e[col] is None:
                    continue
                v, BT = e[col]
                texts = A[col].to_numpy()[rows]
                for i in range(0, len(rows), chunk):
                    C = sp_matmul_topn(v.transform(texts[i:i + chunk]), BT, top_n=k, threshold=thr,
                                       n_threads=N_CPU).tocoo()
                    out.append((rows[i + C.row], e["pos"][C.col], C.data, tag))
        if not out:
            return pd.DataFrame({"a": [], "b": [], "sim_name": [], "sim_addr": []})
        a = np.concatenate([o[0] for o in out]).astype(np.int64)
        b = np.concatenate([o[1] for o in out]).astype(np.int64)
        sim = np.concatenate([o[2] for o in out]).astype(np.float32)
        tag = np.concatenate([np.full(len(o[0]), o[3], np.int8) for o in out])
        df = pd.DataFrame({"key": a * len(self.B) + b,
                           "sim_name": np.where(tag == 0, sim, np.nan),
                           "sim_addr": np.where(tag == 1, sim, np.nan)})
        df = df.groupby("key", sort=False).max().reset_index()
        df["a"] = (df.key // len(self.B)).astype(np.int32)
        df["b"] = (df.key % len(self.B)).astype(np.int32)
        return df.drop(columns="key")

# ---------------------------------------------------------------------
# 4. Features: how similar is each (Source 1, candidate) pair?
# ---------------------------------------------------------------------
FULL = [("tset", fuzz.token_set_ratio), ("tsort", fuzz.token_sort_ratio), ("ratio", fuzz.ratio),
        ("partial", fuzz.partial_ratio), ("jw", JaroWinkler.normalized_similarity)]

def _nums(s):
    return set(re.findall(r"\d+", s))

def features(P, A, B, is_s3):
    a, b = P.a.to_numpy(), P.b.to_numpy()
    g = lambda D, col, idx: D[col].to_numpy()[idx]
    f = pd.DataFrame({"sim_name": P.sim_name.to_numpy(), "sim_addr": P.sim_addr.to_numpy()})
    for col, pre, scorers in (("name_b", "nb", FULL), ("name_sk", "nsk", FULL),
                              ("name_skc", "nskc", [("ratio", fuzz.ratio), ("partial", fuzz.partial_ratio)]),
                              ("addr_sk", "ask", [("tset", fuzz.token_set_ratio), ("ratio", fuzz.ratio),
                                                  ("partial", fuzz.partial_ratio)])):
        x, y = g(A, col, a).tolist(), g(B, col, b).tolist()
        for nm, sc in scorers:
            f[f"{pre}_{nm}"] = cpdist(x, y, scorer=sc, workers=N_CPU).astype(np.float32)
    xa, ya = g(A, "addr_n", a), g(B, "addr_n", b)
    nl = [_nums(s) for s in xa]; nr = [_nums(s) for s in ya]
    f["num_jacc"] = np.array([len(p & q) / len(p | q) if (p | q) else -1 for p, q in zip(nl, nr)], np.float32)
    f["num_contain"] = np.array([len(p & q) / len(q) if q else -1 for p, q in zip(nl, nr)], np.float32)
    sa, sb = g(A, "states", a), g(B, "states", b)
    f["state_rel"] = np.array([-1 if (not p or not q) else (1 if set(p.split("|")) & set(q.split("|")) else 0)
                               for p, q in zip(sa, sb)], np.int8)
    f["name_exact"] = (g(A, "name_skc", a) == g(B, "name_skc", b)).astype(np.int8)
    f["cand_addr_len"] = np.fromiter((len(s) for s in ya), np.int32, len(ya))
    f["s1_addr_len"] = np.fromiter((len(s) for s in xa), np.int32, len(xa))
    f["cand_name_len"] = np.fromiter((len(s) for s in g(B, "name_b", b)), np.int32, len(b))
    f["s1_name_len"] = np.fromiter((len(s) for s in g(A, "name_b", a)), np.int32, len(a))
    f["cand_nonascii"] = g(B, "nonascii", b)
    f["is_s3"] = np.int8(is_s3)
    # context: how this candidate compares with the other candidates of the same Source 1 record
    v = f.nsk_tset.to_numpy() + f.ask_tset.to_numpy()
    grp = pd.DataFrame({"a": a, "v": v}).groupby("a")["v"]
    f["rank_s1"] = grp.rank(ascending=False, method="min").to_numpy(np.float32)
    f["gap_s1"] = (grp.transform("max").to_numpy() - v).astype(np.float32)
    f["n_cand_s1"] = grp.transform("size").to_numpy(np.int32)
    return f

# ---------------------------------------------------------------------
# 5. Scoring (same formula as the leaderboard) and the final decision rule
# ---------------------------------------------------------------------
def macro_f05(pred, truth, ids):
    sc = []
    for s in ids:
        p, t = pred.get(s, set()), truth.get(s, set())
        if not t:
            sc.append(1.0 if not p else 0.0); continue
        tp = len(p & t)
        if tp == 0:
            sc.append(0.0); continue
        P, R = tp / len(p), tp / len(t)
        sc.append(1.25 * P * R / (0.25 * P + R))
    return float(np.mean(sc))

def decide(pairs, thr):
    """pairs: DataFrame(s1_id, cand_id, p). Keep p >= thr; each S2/S3 record goes to at most one S1 (its best)."""
    d = pairs[pairs.p >= thr].sort_values("p", ascending=False).drop_duplicates("cand_id")
    return d.groupby("s1_id")["cand_id"].apply(set).to_dict()

def write_tsv(path, s1_ids, mapping, col):
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(f"source1_entity_id\t{col}\n")
        for s in s1_ids:
            fh.write(s + "\t" + ",".join(sorted(mapping.get(s, ()))) + "\n")


In [ ]:
ROOT = "/kaggle/input"      # the notebook searches here for the .tsv files
WORK = "/kaggle/working"    # model + outputs are saved here
N_TRAIN_S1 = int(os.environ.get("N_TRAIN_S1", 150_000))   # training businesses (same 150k as v1, so scores compare)
BATCH = int(os.environ.get("BATCH", 100_000))             # test businesses processed at once (lower it if memory runs out)
SMOKE = 0                   # >0 only for Claude's small local test: keep this many pool rows
os.makedirs(WORK, exist_ok=True)
log("CPU cores:", N_CPU)


In [ ]:
# =====================================================================
# v4 additions: two extra ways to build the shortlist, on top of v1's same-state TF-IDF shortlist.
#   (a) "embeddings": a small pretrained text model (BAAI/bge-small-en-v1.5, MIT licence, 33M parameters)
#       turns "name , address" into 384 numbers; records that mean the same thing get similar numbers.
#       For every Source 1 record we take the K_EMB closest records in the whole country (no state filter).
#   (b) country-wide TF-IDF on character 3-grams of "name + address", squeezed to 256 numbers (LSA) so the same
#       fast neighbour search works (catches spelling variants that v1's per-state top-15 lists crowd out).
# All candidates are merged with v1's, then scored by LightGBM with v1's features + new similarity features.
# The embedding model is downloaded once from Hugging Face (pretrained weights only; no data leaves Kaggle).
# =====================================================================
EMB_MODEL = os.environ.get("EMB_MODEL", "BAAI/bge-small-en-v1.5")
EMB_BACKEND = os.environ.get("EMB_BACKEND", "torch")      # "onnx" is only used for Claude's local CPU test
K_EMB = int(os.environ.get("K_EMB", 20))                  # embedding neighbours per S1 record and source
K_TF = int(os.environ.get("K_TF", 30))                    # country-wide TF-IDF neighbours per S1 record and source

def emb_text(D):
    return (D.name_b + " , " + D.addr_n).tolist()

_EMB = {}
def _load_emb():
    if _EMB: return _EMB
    if EMB_BACKEND == "onnx":
        import onnxruntime as ort
        from tokenizers import Tokenizer
        tok = Tokenizer.from_file(f"{EMB_MODEL}/tokenizer.json"); tok.enable_truncation(64); tok.enable_padding()
        so = ort.SessionOptions(); so.intra_op_num_threads = N_CPU
        _EMB.update(kind="onnx", tok=tok, sess=ort.InferenceSession(f"{EMB_MODEL}/model_optimized.onnx", so))
    else:
        import torch
        from transformers import AutoTokenizer, AutoModel
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        m = AutoModel.from_pretrained(EMB_MODEL).eval().to(dev)
        if dev == "cuda":
            m = m.half()
            if torch.cuda.device_count() > 1:
                m = torch.nn.DataParallel(m)               # use both T4 GPUs
        _EMB.update(kind="torch", tok=AutoTokenizer.from_pretrained(EMB_MODEL), model=m, dev=dev, torch=torch)
        log("embedding model on", dev, "GPUs:", torch.cuda.device_count())
    return _EMB

def embed(texts, bs=2048):
    """Unit-length float16 vectors, one row per text. Texts are sorted by length so batches need little padding."""
    E = _load_emb()
    order = np.argsort([len(t) for t in texts], kind="stable")
    out = None
    t0 = time.time()
    for n, i in enumerate(range(0, len(texts), bs)):
        idx = order[i:i + bs]; batch = [texts[j] or "-" for j in idx]
        if E["kind"] == "onnx":
            enc = E["tok"].encode_batch(batch)
            ids = np.array([e.ids for e in enc], np.int64); am = np.array([e.attention_mask for e in enc], np.int64)
            v = E["sess"].run(None, {"input_ids": ids, "attention_mask": am, "token_type_ids": np.zeros_like(ids)})[0][:, 0]
        else:
            torch = E["torch"]
            enc = E["tok"](batch, padding=True, truncation=True, max_length=64, return_tensors="pt")
            with torch.inference_mode():
                h = E["model"](input_ids=enc["input_ids"].to(E["dev"]), attention_mask=enc["attention_mask"].to(E["dev"]), return_dict=False)
                v = (h[0] if isinstance(h, tuple) else h.last_hidden_state)[:, 0].float().cpu().numpy()
        v /= np.linalg.norm(v, axis=1, keepdims=True) + 1e-9
        if out is None: out = np.zeros((len(texts), v.shape[1]), np.float16)
        out[idx] = v.astype(np.float16)
        if n % 500 == 499:
            log(f"  embedded {i + len(idx):,}/{len(texts):,} ({(i + len(idx)) / (time.time() - t0):.0f}/s)")
    return out if out is not None else np.zeros((0, 384), np.float16)

def knn(Q, X, k, chunk=512):
    """For each row of Q, the k rows of X with the highest dot product. Returns (rows, cols, sims)."""
    k = min(k, len(X))
    if len(Q) == 0 or k == 0:
        return np.zeros(0, np.int64), np.zeros(0, np.int64), np.zeros(0, np.float32)
    E = _load_emb() if EMB_BACKEND == "torch" else None
    R, C, S = [], [], []
    if E is not None and E["dev"] == "cuda":
        torch = E["torch"]
        Xg = torch.from_numpy(X).cuda()
        for i in range(0, len(Q), chunk):
            s = torch.from_numpy(Q[i:i + chunk]).cuda() @ Xg.T
            v, c = torch.topk(s, k, dim=1)
            R.append(np.repeat(np.arange(i, i + len(c)), k)); C.append(c.cpu().numpy().ravel())
            S.append(v.float().cpu().numpy().ravel())
        del Xg; torch.cuda.empty_cache()
    else:
        Xf = X.astype(np.float32); chunk = max(16, int(2e8 // max(len(X), 1)))   # keep the score block under ~1 GB
        for i in range(0, len(Q), chunk):
            s = Q[i:i + chunk].astype(np.float32) @ Xf.T
            c = np.argpartition(-s, k - 1, axis=1)[:, :k]
            R.append(np.repeat(np.arange(i, i + len(c)), k)); C.append(c.ravel())
            S.append(np.take_along_axis(s, c, 1).ravel())
    return np.concatenate(R), np.concatenate(C), np.concatenate(S).astype(np.float32)

def _tf_text(D):
    return (D.name_skc + " " + D.addr_sk).to_numpy()

def _unit(Z):
    Z = np.asarray(Z, np.float32)
    return (Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-9)).astype(np.float16)

class TfIndex:
    """Country-wide spelling search (no state filter): TF-IDF on character 3-grams of skeleton name + skeleton
    address. An exact sparse search is too slow at this size (about 7 hours on the test set), so each record's
    TF-IDF vector is squeezed to 256 numbers (LSA / truncated SVD) and searched with the same GPU neighbour search.
    Exact TF-IDF cosine is still computed for every final pair as a feature."""
    def __init__(self, B):
        from sklearn.decomposition import TruncatedSVD
        self.v = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 3), dtype=np.float32, sublinear_tf=True, min_df=2)
        self.M = self.v.fit_transform(_tf_text(B))
        dim = max(1, min(256, self.M.shape[1] - 1, self.M.shape[0] - 1))
        fit_rows = np.random.default_rng(0).choice(self.M.shape[0], min(300_000, self.M.shape[0]), replace=False)
        self.svd = TruncatedSVD(dim, n_iter=4, random_state=0).fit(self.M[fit_rows])
        self.Z = _unit(self.svd.transform(self.M))
    def vec(self, A):
        Q = self.v.transform(_tf_text(A))
        return Q, _unit(self.svd.transform(Q))
    def pair_cos(self, QM, a, b):
        """Exact TF-IDF cosine for given pairs: QM = TF-IDF rows of A."""
        out = np.zeros(len(a), np.float32)
        for i in range(0, len(a), 2_000_000):
            out[i:i + 2_000_000] = np.asarray(QM[a[i:i + 2_000_000]].multiply(self.M[b[i:i + 2_000_000]]).sum(1)).ravel()
        return out

def shortlist_v4(Ac, Bc, EA, EB, idx=None, tfi=None):
    """Union of v1's shortlist, embedding neighbours and country-wide TF-IDF neighbours for one country.
    EA / EB: embeddings of Ac / Bc rows. Returns P with columns a, b, sim_name, sim_addr, src_v1, emb_rank, tf_rank."""
    idx = idx or Index(Bc); tfi = tfi or TfIndex(Bc)
    P = idx.query(Ac); P["src_v1"] = np.int8(1)
    r, c, s = knn(EA, EB, K_EMB)
    # knn returns unsorted top-k on CPU; rank them properly by similarity within each query row
    E = pd.DataFrame({"a": r.astype(np.int32), "b": c.astype(np.int32), "s": s})
    E["emb_rank"] = E.groupby("a")["s"].rank(ascending=False, method="first").astype(np.float32)
    QM, ZA = tfi.vec(Ac)
    r, c, s = knn(ZA, tfi.Z, K_TF)
    T = pd.DataFrame({"a": r.astype(np.int32), "b": c.astype(np.int32), "s": s})
    T["tf_rank"] = T.groupby("a")["s"].rank(ascending=False, method="first").astype(np.float32)
    P = P.merge(E[["a", "b", "emb_rank"]], on=["a", "b"], how="outer").merge(T[["a", "b", "tf_rank"]], on=["a", "b"], how="outer")
    P["src_v1"] = P.src_v1.fillna(0).astype(np.int8)
    P["emb_rank"] = P.emb_rank.fillna(K_EMB + 1).astype(np.float32); P["tf_rank"] = P.tf_rank.fillna(K_TF + 1).astype(np.float32)
    P["a"] = P.a.astype(np.int32); P["b"] = P.b.astype(np.int32)
    # similarity features for every pair (v1's sim_name/sim_addr stay empty for pairs v1 did not find)
    a, b = P.a.to_numpy(), P.b.to_numpy()
    P["emb_cos"] = np.einsum("ij,ij->i", EA[a].astype(np.float32), EB[b].astype(np.float32))
    P["tf_cos"] = tfi.pair_cos(QM, a, b)
    return P.reset_index(drop=True)

V4_EXTRA = ["src_v1", "emb_rank", "tf_rank", "emb_cos", "tf_cos", "emb_rank_s1", "emb_gap_s1", "tf_gap_s1"]

def features_v4(P, Ac, Bc, is_s3):
    F = features(P, Ac, Bc, is_s3)
    for c in ("src_v1", "emb_rank", "tf_rank", "emb_cos", "tf_cos"):
        F[c] = P[c].to_numpy()
    g = pd.DataFrame({"a": P.a.to_numpy(), "e": P.emb_cos.to_numpy(), "t": P.tf_cos.to_numpy()}).groupby("a")
    F["emb_rank_s1"] = g["e"].rank(ascending=False, method="min").to_numpy(np.float32)
    F["emb_gap_s1"] = (g["e"].transform("max").to_numpy() - P.emb_cos.to_numpy()).astype(np.float32)
    F["tf_gap_s1"] = (g["t"].transform("max").to_numpy() - P.tf_cos.to_numpy()).astype(np.float32)
    return F


In [ ]:
# ---------------------------------------------------------------------
# v4 TRAINING: 150k training businesses (seed 0, same as v1) vs all Source 2 + Source 3 records.
# Prints shortlist recall (v1 part alone and the v4 union) and "BEST val F0.5" on the same
# 30k held-out businesses v1 used (v1: recall 0.9765, val F0.5 0.9529).
# ---------------------------------------------------------------------
log("loading train Source 1 + ground truth")
s1 = load(ROOT, "train_source1"); gt = load(ROOT, "train_ground_truth")
s1 = s1.sample(N_TRAIN_S1, random_state=0).reset_index(drop=True)
truth = {r.source1_entity_id: set(x for x in r.matched_entity_ids.split(",") if x) for r in gt.itertuples()}
del gt; gc.collect()
A = prepare(s1); log("prepared S1 sample", A.shape)
EA = embed(emb_text(A)); log("embedded S1 sample")

def smoke_pool(B, src):
    """Local test only: shrink the pool to the true matches of our S1 sample plus random other rows."""
    want = set().union(*(truth[s] for s in A.entity_id))
    keep = B.entity_id.isin(want).to_numpy() | (np.random.default_rng(0).random(len(B)) < SMOKE / len(B))
    return B[keep].reset_index(drop=True)

feats, metas = [], []
for src, is_s3 in (("train_source2", 0), ("train_source3", 1)):
    log("loading", src); B = prepare(load(ROOT, src)); log("prepared", src, B.shape)
    if SMOKE: B = smoke_pool(B, src); log("SMOKE pool", B.shape)
    EB = embed(emb_text(B)); log("embedded", src)
    for country in A.country.unique():
        am = (A.country == country).to_numpy(); bm = (B.country == country).to_numpy()
        if not am.any() or not bm.any(): continue
        Ac = A[am].reset_index(drop=True); Bc = B[bm].reset_index(drop=True)
        P = shortlist_v4(Ac, Bc, EA[am], EB[bm]); log(src, country, "pairs", len(P))
        F = features_v4(P, Ac, Bc, is_s3)
        m = pd.DataFrame({"s1_id": Ac.entity_id.to_numpy()[P.a], "cand_id": Bc.entity_id.to_numpy()[P.b],
                          "v1": P.src_v1.to_numpy(), "emb": P.emb_rank.to_numpy() <= K_EMB,
                          "tf": P.tf_rank.to_numpy() <= K_TF})
        m["label"] = [c in truth.get(s, ()) for s, c in zip(m.s1_id, m.cand_id)]
        feats.append(F); metas.append(m); del P; gc.collect()
    del B, EB; gc.collect()
X = pd.concat(feats, ignore_index=True); M = pd.concat(metas, ignore_index=True); del feats, metas; gc.collect()
y = M.label.to_numpy().astype(int)
all_true = sum(len(truth[s]) for s in A.entity_id)
log("pairs", len(M), "per S1", round(len(M) / len(A), 1))
for nm, mask in (("v1 shortlist alone", M.v1.to_numpy() == 1), ("embeddings alone", M.emb.to_numpy()),
                 ("country TF-IDF alone", M.tf.to_numpy()), ("v4 union", np.ones(len(M), bool))):
    log(f"shortlist recall {nm}: {y[mask].sum() / all_true:.4f}  ({mask.sum() / len(A):.1f} pairs per S1)")

rng = np.random.default_rng(42)
val_ids = set(rng.choice(A.entity_id.to_numpy(), int(0.2 * len(A)), replace=False))   # same 30k as v1
iv = M.s1_id.isin(val_ids).to_numpy()
model = lgb.LGBMClassifier(n_estimators=4000, learning_rate=0.05, num_leaves=255, min_child_samples=100,
                           subsample=0.8, subsample_freq=1, colsample_bytree=0.8, n_jobs=N_CPU, verbose=-1)
model.fit(X[~iv], y[~iv], eval_set=[(X[iv], y[iv])], callbacks=[lgb.early_stopping(100, verbose=False)])
log("trees", model.best_iteration_)
pv = M[iv][["s1_id", "cand_id"]].assign(p=model.predict_proba(X[iv])[:, 1])
vl = list(val_ids); vt = {s: truth[s] for s in vl}
log("baseline all-empty", round(macro_f05({}, vt, vl), 4))
log("ceiling (perfect model on this shortlist)",
    round(macro_f05(M[iv & (y == 1)].groupby("s1_id")["cand_id"].apply(set).to_dict(), vt, vl), 4))
res = [(round(t, 2), macro_f05(decide(pv, t), vt, vl)) for t in np.arange(0.20, 0.91, 0.05)]
for t, s in res: log("threshold", t, "val F0.5", round(s, 4))
best_t, best_s = max(res, key=lambda r: r[1])
log("BEST threshold", best_t, "val F0.5", round(best_s, 4), "(v1: 0.9529)")
final = lgb.LGBMClassifier(**{**model.get_params(), "n_estimators": model.best_iteration_})
final.fit(X, y)
final.booster_.save_model(f"{WORK}/model_v4.txt")
json.dump({"threshold": best_t, "val_f05": best_s, "features": list(X.columns)}, open(f"{WORK}/model_v4_meta.json", "w"))
imp = pd.Series(final.feature_importances_, X.columns).sort_values(ascending=False)
log("top features:\n" + imp.head(15).to_string())
del X, M, y, model, final, A, EA, s1, truth, pv, iv; gc.collect()


In [ ]:
# ---------------------------------------------------------------------
# v4 PREDICTION on the test set. Each batch is saved to disk (parts_v4/), so a crash can resume.
# ---------------------------------------------------------------------
PARTS = f"{WORK}/parts_v4"; os.makedirs(PARTS, exist_ok=True)
meta = json.load(open(f"{WORK}/model_v4_meta.json")); booster = lgb.Booster(model_file=f"{WORK}/model_v4.txt")
log("loading test Source 1"); A = prepare(load(ROOT, "test_source1")); log("prepared", A.shape)
print(A.country.value_counts().to_string())
if SMOKE: A = A.sample(min(SMOKE // 10, len(A)), random_state=0).reset_index(drop=True); log("SMOKE S1", A.shape)
EA = embed(emb_text(A)); log("embedded test S1")
s1_ids = A.entity_id.to_numpy()
for src, is_s3 in (("test_source2", 0), ("test_source3", 1)):
    todo = [(c, i) for c in A.country.unique() for i in range(0, int((A.country == c).sum()), BATCH)
            if not os.path.exists(f"{PARTS}/{src}_{c}_{i}.parquet")]
    if not todo:
        log(src, "already done"); continue
    log("loading", src); B = prepare(load(ROOT, src)); log("prepared", src, B.shape)
    if SMOKE: B = B.sample(min(SMOKE, len(B)), random_state=0).reset_index(drop=True); log("SMOKE pool", B.shape)
    EB = embed(emb_text(B)); log("embedded", src)
    for country in A.country.unique():                 # any label works, France included
        a_rows = np.flatnonzero(A.country.to_numpy() == country)
        bm = (B.country == country).to_numpy()
        if not bm.any() or not any(c == country for c, _ in todo): continue
        Bc = B[bm].reset_index(drop=True); EBc = EB[bm]
        idx = Index(Bc); tfi = TfIndex(Bc); bids = Bc.entity_id.to_numpy(); log(src, country, "indexes built")
        for i in range(0, len(a_rows), BATCH):
            out = f"{PARTS}/{src}_{country}_{i}.parquet"
            if os.path.exists(out): continue
            rows = a_rows[i:i + BATCH]; Ac = A.iloc[rows].reset_index(drop=True)
            P = shortlist_v4(Ac, Bc, EA[rows], EBc, idx, tfi)
            F = features_v4(P, Ac, Bc, is_s3)[meta["features"]]
            p = booster.predict(F.to_numpy(np.float32), num_threads=N_CPU).astype(np.float32); del F
            pd.DataFrame({"s1_row": rows[P.a.to_numpy()].astype(np.int32), "cand_id": bids[P.b.to_numpy()],
                          "p": p}).to_parquet(out)
            log(src, country, f"batch {i // BATCH + 1}/{-(-len(a_rows) // BATCH)}", "pairs", len(P))
            del P, p; gc.collect()
        del idx, tfi, Bc, EBc; gc.collect()
    del B, EB; gc.collect()

# ---- combine the saved batches, using little memory (same method as v1.2's combine step) ----
del A, EA; gc.collect()
log("combining batches:", len(glob.glob(f"{PARTS}/*.parquet")), "files,", len(s1_ids), "Source 1 rows")
# 1) Final matches: only pairs above the cut-off are loaded; each S2/S3 record goes to its best S1 only
best = (pl.scan_parquet(f"{PARTS}/*.parquet").filter(pl.col("p") >= meta["threshold"]).collect()
        .sort("p", descending=True).unique(subset="cand_id", keep="first", maintain_order=True))
match = best.group_by("s1_row").agg(pl.col("cand_id").sort())
mline = np.full(len(s1_ids), "", dtype=object)
for r, c in zip(match["s1_row"].to_list(), match["cand_id"].to_list()):
    mline[r] = ",".join(c)
n = int((mline != "").sum()); n_links = best.height; del best, match; gc.collect()
os.makedirs(f"{WORK}/output", exist_ok=True)
with open(f"{WORK}/output/matching_results.tsv", "w", encoding="utf-8") as f:
    f.write("source1_entity_id\tmatched_entity_ids\n")
    for s, m in zip(s1_ids, mline):
        f.write(s + "\t" + m + "\n")
log("done:", len(s1_ids), "rows,", n, "with matches,", len(s1_ids) - n, "empty; avg matches",
    round(n_links / len(s1_ids), 2))
del mline; gc.collect()
# 2) Candidate list, one batch at a time (the Source 2 and Source 3 files of a batch cover the same S1 rows)
cline = np.full(len(s1_ids), "", dtype=object)
for f2 in sorted(glob.glob(f"{PARTS}/*.parquet")):
    if "test_source3_" in f2 and os.path.exists(f2.replace("test_source3_", "test_source2_")): continue
    files = [f2] + [x for x in [f2.replace("test_source2_", "test_source3_")] if x != f2 and os.path.exists(x)]
    g = (pl.concat([pl.read_parquet(x, columns=["s1_row", "cand_id"]) for x in files])
         .group_by("s1_row").agg(pl.col("cand_id").unique(maintain_order=True)))
    for r, c in zip(g["s1_row"].to_list(), g["cand_id"].to_list()):
        cline[r] = ",".join(c)
    del g; gc.collect()
with open(f"{WORK}/output/candidate_pairs.tsv", "w", encoding="utf-8") as f:
    f.write("source1_entity_id\tcandidate_entity_ids\n")
    for s, c in zip(s1_ids, cline):
        f.write(s + "\t" + c + "\n")
log("candidate file written")


In [ ]:
# candidate_pairs check is skipped (none.tsv): it needs a lot of memory
validator = glob.glob(f'{ROOT}/**/validate_submission.py', recursive=True)[0]
test_dir = os.path.dirname(find_file(ROOT, 'test_source1')[0])
!python3 {validator} --matching {WORK}/output/matching_results.tsv --candidate none.tsv --test-dir {test_dir}

**Done.** Open this version's **Output** tab, go into `output/`, and download **`matching_results.tsv`**. Keep `candidate_pairs.tsv` for the final zip. You can delete the `parts_v4/` folder.